# Install lirary


In [1]:
!pip install -r requirements.txt

# Import Libary

In [2]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from itables import show
import itertools

import nest_asyncio
import asyncio
import pandas as pd
import sys

nest_asyncio.apply()  # Patch Jupyter's asyncio loop

async def run_command(
    cmd,
    cwd=None,
    timeout=None,
    print_realtime=False
):
    """Run a shell command asynchronously with high performance.

    Args:
        cmd (str): The shell command to run.
        cwd (str): Optional working directory.
        timeout (float): Timeout in seconds.
        print_realtime (bool): If True, prints stdout/stderr line by line.

    Returns:
        dict: {
            "returncode": int,
            "stdout": str,
            "stderr": str,
            "command": str,
        }
    """
    proc = await asyncio.create_subprocess_shell(
        cmd,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
        cwd=cwd
    )

    stdout, stderr = [], []

    try:
        async def read_stream(stream, buffer, stream_name):
            while True:
                line = await stream.readline()
                if line:
                    decoded = line.decode(errors="replace")
                    buffer.append(decoded)
                    if print_realtime:
                        print(f"[{stream_name}] {decoded}", end="")
                else:
                    break

        await asyncio.wait_for(
            asyncio.gather(
                read_stream(proc.stdout, stdout, "STDOUT"),
                read_stream(proc.stderr, stderr, "STDERR"),
                proc.wait()
            ),
            timeout=timeout
        )

    except asyncio.TimeoutError:
        proc.kill()
        return {
            "returncode": -1,
            "stdout": "".join(stdout),
            "stderr": "".join(stderr) + "\n[Timeout]",
            "command": cmd,
        }

    return {
        "returncode": proc.returncode,
        "stdout": "".join(stdout),
        "stderr": "".join(stderr),
        "command": cmd,
    }

def display_data(df):

    # --- Pivot to compare BRRT vs BRRT_Optimize ---
    pivot_time = df.pivot(index="run", columns="algorithm", values="search_time")
    pivot_length = df.pivot(index="run", columns="algorithm", values="path_length")
    pivot_nodes = df.pivot(index="run", columns="algorithm", values="node_count")
    pivot_success = df.pivot(index="run", columns="algorithm", values="success")
    pivot_idx = df.pivot(index="run", columns="algorithm", values="num_iterations")

    # --- Display deep-inspect comparison table ---


    # show(display_df, max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")
    # --- Plot line charts for per-run comparison ---
    fig, axs = plt.subplots(4, 1, figsize=(10, 10), sharex=True)

    pivot_time.plot(ax=axs[0], marker='o', title="Search Time per Run")
    axs[0].set_ylabel("Time (s)")
    axs[0].grid(True)

    pivot_length.plot(ax=axs[1], marker='o', title="Path Length per Run")
    axs[1].set_ylabel("Length")
    axs[1].grid(True)

    pivot_nodes.plot(ax=axs[2], marker='o', title="Node Count per Run")
    axs[2].set_ylabel("Nodes")
    axs[2].set_xlabel("Run")
    axs[2].grid(True)


    pivot_idx.plot(ax=axs[3], marker='o', title="Iterations Count per Run")
    axs[3].set_ylabel("Interations")
    axs[3].set_xlabel("Run")
    axs[3].grid(True)

    plt.tight_layout()
    plt.show()
    return df


# Load all files into a list of dictionaries
def summary_statistics(raw_data):
    records = []
    for run_entry in raw_data["results"]:
        run = run_entry["run"]
        for algo_name, result in run_entry["algorithms"].items():
            records.append({
                "run": run,
                "algorithm": algo_name,
                "search_time": result["search_time"],
                "path_length": result["path_length"],
                "node_count": result["node_count"],
                "num_iterations": result.get("num_iterations", 0),
                "success": result["success"],
                "start": result["start"],
                "goal": result["goal"],
            })

    df = pd.DataFrame(records)
    # print(df)
    # --- Display deep-inspect comparison table ---


    agg = df.groupby("algorithm").agg(
        success_rate=("success", "mean"),
        avg_search_time=("search_time", "mean"),
        avg_path_length=("path_length", "mean"),
        avg_node_count=("node_count", "mean"),
        avg_iterations=("num_iterations", "mean")
    ).round(6)
    # show(agg)
    return df,agg

def summary_statistics_single(raw_data):
    records = []
    for run_entry in raw_data["results"]:
        run = run_entry["run"]
        for algo_name, result in run_entry["algorithms"].items():
            records.append({
                "run": run,
                "algorithm": algo_name,
                "search_time": result["search_time"],
                "path_length": result["path_length"],
                "node_count": result["node_count"],
                "num_iterations": result.get("num_iterations", 0),
                "success": result["success"],
                "start": result["start"],
                "goal": result["goal"],
            })

    df = pd.DataFrame(records)
    # print(df)
    # --- Display deep-inspect comparison table ---


    agg = df.groupby("algorithm").agg(
        success_rate=("success", "mean"),
        avg_search_time=("search_time", "mean"),
        avg_path_length=("path_length", "mean"),
        avg_node_count=("node_count", "mean"),
        avg_iterations=("num_iterations", "mean")
    ).round(6)

    sum = df.groupby("algorithm")[["search_time","path_length","node_count","num_iterations"]].sum().round(6)
    # show(agg)
    return df,agg,sum

def load_a_data(output_path):
    with open(output_path, 'r') as f:
        content = json.load(f)
        print("Parameters:", content["parameters"])
        sum_df, agg,sum = summary_statistics_single(content)
        show(sum, max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")
        show(agg, max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")
        show (sum_df, max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")
        display_data(sum_df)
        

# AUTO TURNING

In [ ]:
# !bash ./build.sh
# !source devel/setup.bash && roslaunch path_finder test_planners.launch

Base path: /home/xuanloc/DACN/ICIT/brrt_optimize
Source space: /home/xuanloc/DACN/ICIT/brrt_optimize/src
Build space: /home/xuanloc/DACN/ICIT/brrt_optimize/build
Devel space: /home/xuanloc/DACN/ICIT/brrt_optimize/devel
Install space: /home/xuanloc/DACN/ICIT/brrt_optimize/install
####
#### Running command: "make cmake_check_build_system" in "/home/xuanloc/DACN/ICIT/brrt_optimize/build"
####
-- Using CATKIN_DEVEL_PREFIX: /home/xuanloc/DACN/ICIT/brrt_optimize/devel
-- Using CMAKE_PREFIX_PATH: /opt/ros/noetic
-- This workspace overlays: /opt/ros/noetic
-- Found PythonInterp: /home/xuanloc/miniconda3/bin/python3 (found suitable version "3.13.5", minimum required is "3") 
-- Using PYTHON_EXECUTABLE: /home/xuanloc/miniconda3/bin/python3
-- Using Debian Python package layout
-- Using empy: /home/xuanloc/miniconda3/lib/python3.13/site-packages/em.py
-- Using CATKIN_ENABLE_TESTING: ON
-- Call enable_testing()
-- Using CATKIN_TEST_RESULTS_DIR: /home/xuanloc/DACN/ICIT/brrt_optimize/build/test_resu

# Generate test parameter

In [ ]:
# import random
# import time
# import itertools # ADDED: Required for the loop
# import json      # ADDED: Required to save files
# import os

# # Define value sets
# map_size = 100
# # Corrected typo 'espsilon' -> 'epsilon'
# epsilon_values = [map_size/200, map_size/100, map_size/50, map_size/10] 
# p1_values = [0.8, 0.5, 0.3]
# u_p_values = [0, 0.5, 1, 3, 12]

# alpha_values = [0, 1, 3, 6, 10]
# beta_values = [0, 1, 4, 7]
# gamma_values = [0, 1, 5, 7]

# configs = []
# trial = 1

# # MODIFIED: Updated path to match your environment structure
# input_dir = "/home/xuanloc/DACN/ICIT/brrt_optimize/experiments/input/"

# # create input directory if it doesn't exist
# if not os.path.exists(input_dir):
#     os.makedirs(input_dir)
#     print(f"Created directory: {input_dir}")

# print(f"Generating configuration files in: {input_dir}")

# # Loop over p1 and u_p
# for epsilon, p1, u_p, alpha, beta, gamma in itertools.product(epsilon_values, p1_values, u_p_values, alpha_values, beta_values, gamma_values):
    
#     # Progress print (every 100 trials to reduce clutter)
#     if trial % 100 == 0:
#         print(f"Generating trial {trial}...")

#     environment = f"{epsilon:.2f}_{p1}_{u_p}_{round(alpha,2)}_{round(beta,2)}_{round(gamma,2)}"
    
#     config = {
#         "trial": trial,
#         "environment": environment,
#         "p1": p1,
#         "u_p": u_p,
#         "alpha": round(alpha, 2),
#         "beta": round(beta, 2),
#         "gamma": round(gamma, 2),
#         "epsilon": epsilon,
#         # Add trap parameters here if you want to test them dynamically later
#         "trap_h_threshold": 0.01,
#         "trap_count_limit": 20,
#         "trap_step_limit": 50
#     }
    
#     input_path = input_dir + f"brrt_{trial}.json" # Simplified name to just trial number for easier looping
    
#     with open(input_path, "w") as f:
#         json.dump(config, f, indent=4)
    
#     trial += 1
#     # Removed time.sleep(0.1) - it will take too long to generate ~4800 files with sleep. 
#     # Modern OS handles file creation fast enough.

# print(f"Done! Generated {trial-1} configuration files.")

Created directory: /home/xuanloc/DACN/ICIT/brrt_optimize/experiments/input/
Generating configuration files in: /home/xuanloc/DACN/ICIT/brrt_optimize/experiments/input/
Generating trial 100...
Generating trial 200...
Generating trial 300...
Generating trial 400...
Generating trial 500...
Generating trial 600...
Generating trial 700...
Generating trial 800...
Generating trial 900...
Generating trial 1000...
Generating trial 1100...
Generating trial 1200...
Generating trial 1300...
Generating trial 1400...
Generating trial 1500...
Generating trial 1600...
Generating trial 1700...
Generating trial 1800...
Generating trial 1900...
Generating trial 2000...
Generating trial 2100...
Generating trial 2200...
Generating trial 2300...
Generating trial 2400...
Generating trial 2500...
Generating trial 2600...
Generating trial 2700...
Generating trial 2800...
Generating trial 2900...
Generating trial 3000...
Generating trial 3100...
Generating trial 3200...
Generating trial 3300...
Generating trial

start x: 27.7002 y: 28.315 z: 0.399877
goal x: -46.8343 y: -20.9334 z: 0.11905
start x: -46.8343 y: -20.9334 z: 0.11905
goal x: -1.93097 y: -31.3365 z: -0.364647
start x: -1.93097 y: -31.3365 z: -0.364647
goal x: -46.47 y: 16.7593 z: 0.488246
start x: -46.47 y: 16.7593 z: 0.488246
goal x: -3.18236 y: 42.2701 z: 0.364305
start x: -3.18236 y: 42.2701 z: 0.364305
goal x: 32.2084 y: 22.0208 z: -0.230854
start x: 32.2084 y: 22.0208 z: -0.230854
goal x: 49.3568 y: -10.1539 z: 0.480099
start x: 49.3568 y: -10.1539 z: 0.480099
goal x: 12.1196 y: -40.5509 z: 0.0306626
start x: 12.1196 y: -40.5509 z: 0.0306626
goal x: 49.3788 y: -41.8464 z: -0.107417
start x: 49.3788 y: -41.8464 z: -0.107417
goal x: 41.3253 y: -20.1565 z: -0.0347332
start x: 41.3253 y: -20.1565 z: -0.0347332
goal x: 46.0132 y: 49.4317 z: 0.0919143
start x: 46.0132 y: 49.4317 z: 0.0919143
goal x: 27.0913 y: 23.5739 z: -0.174355
start x: 27.0913 y: 23.5739 z: -0.174355
goal x: -37.0081 y: -48.2243 z: -0.110512
start x: -37.0081 y:

In [ ]:
# import os
# import time
# import subprocess

# # 1. Define the correct root path for your environment
# workspace_root = "/home/xuanloc/DACN/ICIT/brrt_optimize"

# # 2. Set up directories relative to the root
# output_dir = os.path.join(workspace_root, "experiments/output-20/")
# input_dir =  os.path.join(workspace_root, "experiments/input/")
# setup_bash = os.path.join(workspace_root, "devel/setup.bash")

# # Create output directory if it doesn't exist
# if not os.path.exists(output_dir):
#     os.makedirs(output_dir)

# files_input = set(os.listdir(input_dir))
# if os.path.exists(output_dir):
#     files_output = set(os.listdir(output_dir))
# else:
#     files_output = set()

# only_in_input = files_input - files_output
# print(f"Files in A not in B: {len(only_in_input)}")

# for f in sorted(only_in_input):
#     print(f)
    
#     # Construct full paths for input and output files
#     input_path = os.path.join(input_dir, f)
#     output_path = os.path.join(output_dir, f)

#     # 3. Construct command using the CORRECT setup.bash path
#     cmd = f"source {setup_bash} && roslaunch path_finder test_planners.launch input_param:={input_path} output_result:={output_path}"
    
#     print(f"Running command: {cmd}")
    
#     # 4. Execute command using subprocess (works in terminal)
#     try:
#         subprocess.run(cmd, shell=True, check=True, executable='/bin/bash')
#     except subprocess.CalledProcessError as e:
#         print(f"Error occurred processing {f}: {e}")
        
#     time.sleep(1)

Files in A not in B: 4800
brrt_1.json
Running command: source /home/xuanloc/DACN/ICIT/brrt_optimize/devel/setup.bash && roslaunch path_finder test_planners.launch input_param:=/home/xuanloc/DACN/ICIT/brrt_optimize/experiments/input/brrt_1.json output_result:=/home/xuanloc/DACN/ICIT/brrt_optimize/experiments/output-20/brrt_1.json
... logging to /home/xuanloc/.ros/log/d4f2ccee-d424-11f0-af0e-ffa602a12538/roslaunch-xuanloc-28773.log
Checking log directory for disk usage. This may take a while.
Press Ctrl-C to interrupt
]2;/home/xuanloc/DACN/ICIT/brrt_optimize/src/path_finder/launch/test_planners.launch
started roslaunch server http://xuanloc:41879/

SUMMARY

PARAMETERS
 * /path_finder_node/BRRT/max_iteration: 100000
 * /path_finder_node/BRRT/max_tree_node_nums: 200000
 * /path_finder_node/BRRT/search_time: 1.0
 * /path_finder_node/BRRT/steer_length: 1.0
 * /path_finder_node/BRRT/step: 0.5
 * /path_finder_node/BRRT_Optimize/alpha: 7.0
 * /path_finder_node/BRRT_Optimize/beta: 2.0
 * /path

It's recommended that you use the 'rosclean' command.


ROS_MASTER_URI=http://localhost:11311
]2;/home/xuanloc/DACN/ICIT/brrt_optimize/src/path_finder/launch/test_planners.launch http://localhost:11311
setting /run_id to d4f2ccee-d424-11f0-af0e-ffa602a12538
process[rosout-1]: started with pid [28809]
started core service [/rosout]
process[random_forest-2]: started with pid [28816]
process[path_finder_node-3]: started with pid [28817]
map initialized: 
[INFO] [1765191511.006828414]: Selected map type: random_large
number of ostacle125cloudMap.points.size() = 136408
input_param: /home/xuanloc/DACN/ICIT/brrt_optimize/experiments/input/brrt_1.json
output_result: /home/xuanloc/DACN/ICIT/brrt_optimize/experiments/output-20/brrt_1.json


[WARN] [1765191510.985655442]: [RRT#] param: steer_length: 1
[WARN] [1765191510.986536778]: [RRT#] param: search_radius: 60
[WARN] [1765191510.986611528]: [RRT#] param: search_time: 1
[WARN] [1765191510.986629570]: [RRT#] param: max_tree_node_nums: 200000
[WARN] [1765191510.986641945]: [RRT#] param: use_informed_sampling: 1
[WARN] [1765191510.986655195]: [RRT#] param: use_GUILD_sampling: 0
[WARN] [1765191510.994520757]: [RRT*] param: steer_length: 1
[WARN] [1765191510.994628549]: [RRT*] param: search_radius: 60
[WARN] [1765191510.994649632]: [RRT*] param: search_time: 1
[WARN] [1765191510.994666841]: [RRT*] param: max_tree_node_nums: 200000
[WARN] [1765191510.994682549]: [RRT*] param: use_informed_sampling: 1
[WARN] [1765191510.994697132]: [RRT*] param: use_GUILD_sampling: 0
[WARN] [1765191511.004578949]: [RRT] param: steer_length: 1
[WARN] [1765191511.004822950]: [RRT] param: search_radius: 60
[WARN] [1765191511.004903034]: [RRT] param: search_time: 1
[WARN] [1765191511.004946867]: [R

[INFO] [1765191512.076987179]: no map rcved yet.
glb occ set


[WARN] [1765191513.076840020]: Timer tick


[INFO] [1765191513.076885353]: Running Test 1
[INFO] [1765191513.076890645]: Running Test 1 of 100
start x: -37.5613 y: -21.2501 z: 0.497143
goal x: -20.0388 y: -4.61838 z: 0.101691
[INFO] [1765191513.399571650]: Running Test 2 of 100
start x: -20.0388 y: -4.61838 z: 0.101691
goal x: 48.0559 y: 33.5852 z: -0.136093
[INFO] [1765191513.586584635]: Running Test 3 of 100
start x: 48.0559 y: 33.5852 z: -0.136093
goal x: -8.95506 y: -2.27397 z: -0.0381799
[INFO] [1765191513.680875505]: Running Test 4 of 100
start x: -8.95506 y: -2.27397 z: -0.0381799
goal x: 46.1514 y: -14.5521 z: -0.018432
[INFO] [1765191513.753316485]: Running Test 5 of 100
start x: 46.1514 y: -14.5521 z: -0.018432
goal x: -0.648059 y: -47.3427 z: 0.0741572
[INFO] [1765191513.767441397]: Running Test 6 of 100
start x: -0.648059 y: -47.3427 z: 0.0741572
goal x: -4.77508 y: 48.476 z: 0.158112
[INFO] [1765191513.968533044]: Running Test 7 of 100
start x: -4.77508 y: 48.476 z: 0.158112
goal x: 5.10524 y: -22.4014 z: 0.182719
[

KeyboardInterrupt: 

# Read all files in the directory

In [3]:
import json
import glob
import pandas as pd
def load_a_file(file_path):
    with open(file_path, 'r') as f:
        content = json.load(f)    
        records = []
        for run_entry in content["results"]:
            run = run_entry["run"]
            for algo_name, result in run_entry["algorithms"].items():
                records.append({
                    "run": run,
                    "algorithm": algo_name,
                    "search_time": result["search_time"],
                    "path_length": result["path_length"],
                    "node_count": result["node_count"],
                    "num_iterations": result.get("num_iterations", 1) +1,
                    "success": result["success"],
                    # "start": result["start"],
                    # "goal": result["goal"],
                })

        return pd.DataFrame(records), content["parameters"]
def compare_two_algorithms_success(df):
    # Filter for BRRT and BRRT_Optimize
    brrt_data = df[df['algorithm'] == 'BRRT']
    brrt_opt_data = df[df['algorithm'] == 'BRRT_Optimize']

    # Merge on 'run' to compare success rates
    merged = pd.merge(brrt_data[['run', 'success']], brrt_opt_data[['run', 'success']], on='run', suffixes=('_brrt', '_brrt_opt'))

    # Calculate success rate for each algorithm
    success_rate_brrt = merged['success_brrt'].mean()
    success_rate_brrt_opt = merged['success_brrt_opt'].mean()

    # Filter only successful runs
    df_success = df[df['success'] == True]

    # Count how many algorithms succeeded per run
    success_counts = df_success.groupby('run')['algorithm'].nunique()

    # Keep only runs where both BRRT and BRRT_Optimize succeeded
    valid_runs = success_counts[success_counts == 2].index
    df_valid = df_success[df_success['run'].isin(valid_runs)]

    # Now pivot and compare safely
    df_compare = df_valid.pivot(index='run', columns='algorithm', values=['search_time', 'path_length', 'node_count', 'num_iterations'])

    # Drop any remaining NaNs (safety)
    # df_compare = df_compare.dropna()

    # Difference and summary
    df_diff = pd.DataFrame({
        'search_time': df_compare[('search_time', 'BRRT_Optimize')] - df_compare[('search_time', 'BRRT')],
        'path_length': df_compare[('path_length', 'BRRT_Optimize')] -  df_compare[('path_length', 'BRRT')],
        'node_count': df_compare[('node_count', 'BRRT_Optimize')] - df_compare[('node_count', 'BRRT')],
        'num_iterations': df_compare[('num_iterations', 'BRRT_Optimize')] - df_compare[('num_iterations', 'BRRT')],
    })

    return success_rate_brrt_opt, success_rate_brrt, df_diff
    # df_mean_diff = df_diff.mean()
    # print("=== Mean Metric Differences (BRRT_Optimize - BRRT) ===")
    # print(df_mean_diff)
    # # df_mean_diff = df_diff.mean()

    # # print("=== Mean Metric Differences (BRRT_Optimize - BRRT, Only if Both Success) ===")
    # print(df_compare)

def compare_algorithms_success(df,number_of_algorithms=2):

    # Filter only successful runs

    df_success = df[df['success'] == True]
    count_algo_success = df.groupby('algorithm').agg(
        success_rate=("success", "mean"))
    # Count how many algorithms succeeded per run
    success_counts = df_success.groupby('run')['algorithm'].nunique()

    # Keep only runs where both BRRT and BRRT_Optimize succeeded
    valid_runs = success_counts[success_counts == number_of_algorithms].index
    df_valid = df_success[df_success['run'].isin(valid_runs)]

    # Now pivot and compare safely
    df_compare = df_valid.pivot(index='run', columns='algorithm', values=['search_time', 'path_length', 'node_count', 'num_iterations'])


    return df_compare,count_algo_success


# Compare 1 file


In [ ]:
# ! . build.sh
# from itables import show
# import json

# config = {
#     "trial": 1 ,
#     "environment": f"test 2 cases",
#     "p1": 1.0,
#     "u_p": 2.0,
#     "alpha": 7.0,
#     "beta": 1.0,
#     "gamma": 1.0,
#     "epsilon": 1.0,
# }
# input_path = "/home/x/Develop/brrt_optimize/brrt_input.json"
# output_path = "/home/x/Develop/brrt_optimize/brrt_output_40.json"
# with open(input_path, "w") as f:
#     json.dump(config, f, indent=4)
# cmd = f"source ./devel/setup.bash && roslaunch path_finder test_planners.launch input_param:={input_path} output_result:={output_path}"
# ! {cmd}





Base path: /home/xuanloc/DACN/ICIT/brrt_optimize
Source space: /home/xuanloc/DACN/ICIT/brrt_optimize/src
Build space: /home/xuanloc/DACN/ICIT/brrt_optimize/build
Devel space: /home/xuanloc/DACN/ICIT/brrt_optimize/devel
Install space: /home/xuanloc/DACN/ICIT/brrt_optimize/install
####
#### Running command: "make cmake_check_build_system" in "/home/xuanloc/DACN/ICIT/brrt_optimize/build"
####
####
#### Running command: "make -j6 -l6" in "/home/xuanloc/DACN/ICIT/brrt_optimize/build"
####
[  0%] Built target std_msgs_generate_messages_eus
[  0%] Built target geometry_msgs_generate_messages_eus
[  0%] Built target _self_msgs_and_srvs_generate_messages_check_deps_GlbObsRcv
[  0%] Built target std_msgs_generate_messages_nodejs
[  0%] Built target _self_msgs_and_srvs_generate_messages_check_deps_output_point
[  0%] Built target _self_msgs_and_srvs_generate_messages_check_deps_LearningSampler
[  0%] Built target geometry_msgs_generate_messages_nodejs
[  0%] Built target _self_msgs_and_srvs_gener

FileNotFoundError: [Errno 2] No such file or directory: '/home/x/Develop/brrt_optimize/brrt_input.json'

In [5]:
output = "/home/xuanloc/DACN/ICIT/brrt_optimize/eval/output.json"
data,config = load_a_file(output)
compared , cout_success = compare_algorithms_success(data,number_of_algorithms=3)
unstacked = compared.mean().unstack()
print(config)
show(data.groupby("algorithm").sum(), max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")
show(cout_success, max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")
show(unstacked, max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")
show(compared, max_rows=20, max_columns=10, table_classes="table table-striped table-bordered")


{'alpha': 2.0, 'beta': 8.0, 'epsilon': 1.0, 'gamma': 8.0, 'p1': 0.7, 'u_p': 0.0}


/home/xuanloc/miniconda3/envs/ros/lib/python3.10/site-packages/itables/typing.py:201: SyntaxWarning: These arguments are not documented in ITableOptions: {'table_classes', 'max_rows', 'max_columns'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(
/home/xuanloc/miniconda3/envs/ros/lib/python3.10/site-packages/itables/typing.py:201: SyntaxWarning: These arguments are not documented in DTForITablesOptions: {'table_classes', 'max_rows', 'max_columns'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(


Loading ITables v2.4.3 from the internet... (need help?)


/home/xuanloc/miniconda3/envs/ros/lib/python3.10/site-packages/itables/typing.py:201: SyntaxWarning: These arguments are not documented in ITableOptions: {'table_classes', 'max_rows', 'max_columns'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(
/home/xuanloc/miniconda3/envs/ros/lib/python3.10/site-packages/itables/typing.py:201: SyntaxWarning: These arguments are not documented in DTForITablesOptions: {'table_classes', 'max_rows', 'max_columns'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(


Loading ITables v2.4.3 from the internet... (need help?)


/home/xuanloc/miniconda3/envs/ros/lib/python3.10/site-packages/itables/typing.py:201: SyntaxWarning: These arguments are not documented in ITableOptions: {'table_classes', 'max_rows', 'max_columns'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(
/home/xuanloc/miniconda3/envs/ros/lib/python3.10/site-packages/itables/typing.py:201: SyntaxWarning: These arguments are not documented in DTForITablesOptions: {'table_classes', 'max_rows', 'max_columns'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(


Loading ITables v2.4.3 from the internet... (need help?)


/home/xuanloc/miniconda3/envs/ros/lib/python3.10/site-packages/itables/typing.py:201: SyntaxWarning: These arguments are not documented in ITableOptions: {'table_classes', 'max_rows', 'max_columns'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(
/home/xuanloc/miniconda3/envs/ros/lib/python3.10/site-packages/itables/typing.py:201: SyntaxWarning: These arguments are not documented in DTForITablesOptions: {'table_classes', 'max_rows', 'max_columns'}. You can silence this warning by setting `itables.options.warn_on_undocumented_option=False`. If you believe ITableOptions should be updated, please make a PR or open an issue at https://github.com/mwouts/itables
  warnings.warn(


Loading ITables v2.4.3 from the internet... (need help?)


In [11]:


file_paths = glob.glob("/home/x/brrt_optimize/experiments/20250709/output-10/*")
cout = 0
cout_success = 0
cout_path_length = 0
cout_node_count = 0
cout_iterations = 0
count_case1 = 0
count_case2 = 0
for file_path in file_paths:
    data,config = load_a_file(file_path)
    
    compared , cout_success = compare_algorithms_success(data,number_of_algorithms=3)
    unstacked = compared.mean().unstack()
    BRRT_rate = cout_success["success_rate"]["BRRT"]
    CASE1_rate = cout_success["success_rate"]["BRRT_Case1"]
    CASE2_rate = cout_success["success_rate"]["BRRT_Case2"]
    BRRT_search_time = unstacked["BRRT"]["search_time"]
    CASE1_search_time = unstacked["BRRT_Case1"]["search_time"]
    CASE2_search_time = unstacked["BRRT_Case2"]["search_time"]
    if CASE1_rate > BRRT_rate and CASE1_rate > CASE2_rate:
        count_case1 += 1
        if CASE1_search_time < BRRT_search_time:
            print("********************************************")
            print("BRRT_Case1 outperforms BRRT, BRRT_Case2")
            print(config)
            print(unstacked)
            print(cout_success)
            print("********************************************")
        elif BRRT_search_time > CASE2_search_time and CASE2_rate > BRRT_rate:
            print("********************************************")
            print("BRRT_Case2 outperforms search time BRRT_Case1 but not success rate")
            print(config)
            print(unstacked)
            print(cout_success)
            print("********************************************")
        # else:
        #     print("==========================================")
        #     print("BRRT_Case1 outperforms BRRT, BRRT_Case2 but with longer search time")
        #     print(config)
        #     print(unstacked)
        #     print(cout_success) 
        #     print("==========================================")

    elif CASE2_rate > BRRT_rate and CASE2_rate > CASE1_rate:
        count_case2 += 1
        if CASE2_search_time < BRRT_search_time:
            print("********************************************")
            print("BRRT_Case2 outperforms BRRT, BRRT_Case1")
            print(config)
            print(unstacked)
            print(cout_success)
            print("********************************************")
        elif CASE1_search_time < BRRT_search_time and CASE1_rate > BRRT_rate:
            print("********************************************")
            print("BRRT_Case1 outperforms time BRRT_Case2 but not success rate")
            print(config)
            print(unstacked)
            print(cout_success)
            print("********************************************")
        # else:
        #     print("==========================================")
        #     print("BRRT_Case2 outperforms BRRT, BRRT_Case1 but with longer search time")
        #     print(config)
        #     print(unstacked)
        #     print(cout_success)
        #     print("==========================================")

    
#     print(f"BRRT_rate: {BRRT_rate}, CASE1_rate: {CASE1_rate}, CASE2_rate: {CASE2_rate}")
    # if CASE1_rate > BRRT_rate:
    #     if CASE2_rate > CASE1_rate:
    #         count_case2 += 1
    #         print("==========================================")
    #         print("BRRT_Case2 outperforms BRRT, BRRT_Case1")
    #         print(config)
    #         print(unstacked)
    #         print(cout_success)
    #         print("==========================================")
    #     else:
    #         count_case1 += 1
    #         print("==========================================")
    #         print("BRRT_Case1 outperforms BRRT, BRRT_Case2")
    #         print(config)
    #         print(unstacked)
    #         print(cout_success)
    #         print("==========================================")
print(f"Total parameter settings where BRRT_Case1 best success: {count_case1}")           
print(f"Total parameter settings where BRRT_Case2 best success: {count_case2}")     
#     # if unstacked[]
#     # success_rate_brrt_opt, success_rate_brrt, diff =  compare_two_algorithms_success(data)
#     # success_rate = success_rate_brrt_opt - success_rate_brrt
#     # mean= diff.mean()
#     # if success_rate >= 0:
#     #     if mean["search_time"] < 0:
#     #         print ("----------------------------------------------")
#     #         print(success_rate_brrt_opt, success_rate_brrt)
#     #         print(f"Configuration: {config}")
#     #         print(f"Diff Success Rate: {success_rate:.2f}")
#     #         print(diff.mean())
#     #         print ("----------------------------------------------")
#     #         cout += 1
#     # if success_rate >= 0:
#     #     cout_success += 1
#     # if mean["path_length"] < 0:
#     #     cout_path_length += 1
#     # if mean["node_count"] < 0:
#     #     cout_node_count += 1
#     # if mean["num_iterations"] < 0:
#     #     cout_iterations += 1

#     break
# # print(f"Total parameter settings where BRRT_Optimize outperforms BRRT overall: {cout}")
# # print(f"Total parameter settings where BRRT_Optimize was success better: {cout_success}")
# # print(f"Total parameter settings where BRRT_Optimize achieved a shorter path length:  {cout_path_length}")
# # print(f"Total parameter settings where BRRT_Optimize used fewer nodes: {cout_node_count}")
# # print(f"Total parameter settings where BRRT_Optimize required fewer iterations:  {cout_iterations}")


Total parameter settings where BRRT_Case1 best success: 0
Total parameter settings where BRRT_Case2 best success: 0


In [12]:
import glob
import json
file_paths = glob.glob("/home/x/brrt_optimize/experiments/20250709/output-10/*")
data_agg = {}
data_summary_df = {}
for path in file_paths:
    with open(path, 'r') as f:
        content = json.load(f)
        sum_df, agg = summary_statistics(content)
        data_agg[path] = agg
        data_summary_df[path] = sum_df

In [13]:
print(len(file_paths))

0


# Select the best

In [14]:



def select_the_better_han(all_data:dict):
    best_all = 1000000
    best_all_file = None

    best_time = 1000000
    best_time_file = None

    best_path_length = 1000000
    best_path_length_file = None    

    best_node_count = 1000000
    best_node_count_file = None

    best_iterations = 1000000
    best_iterations_file = None
    for filepath, df in all_data.items():
        if (df.loc['BRRT_Optimize']["success_rate"] < df.loc['BRRT']["success_rate"]):
            continue
        mul = df.loc['BRRT_Optimize']/ df.loc['BRRT']
        percent = mul [['avg_search_time', 'avg_path_length','avg_node_count', 'avg_iterations']]
        # print(f"File: {filepath}",percent)
        sum_percent = percent.sum().sum()
        if (sum_percent < best_all):
            best_all_file = filepath
            best_all = sum_percent
        if (percent['avg_search_time'] < best_time):
            best_time_file = filepath
            best_time = percent['avg_search_time']
     
        if (percent['avg_path_length'] < best_path_length):
            best_path_length_file = filepath
            best_path_length = percent['avg_path_length']
        if (percent['avg_node_count'] < best_node_count):
            best_node_count_file = filepath
            best_node_count = percent['avg_node_count']
        if (percent['avg_iterations'] < best_iterations):
            best_iterations_file = filepath
            best_iterations = percent['avg_iterations']
    return {
        "best_all_file": best_all_file,
        "best_all": best_all,
        "best_time_file": best_time_file,
        "best_time": best_time,
        "best_path_length_file": best_path_length_file,
        "best_path_length": best_path_length,
        "best_node_count_file": best_node_count_file,
        "best_node_count": best_node_count,
        "best_iterations_file": best_iterations_file,
        "best_iterations": best_iterations
    }
summary = select_the_better_han(data_agg)
 
print(f"best_all_file File: {summary['best_all_file']}, Score: {summary['best_all']}")

load_a_data(summary["best_all_file"])

print(f"best_time_file File: {summary['best_time_file']}, Score: {summary['best_time']}")
load_a_data(summary["best_time_file"])
print(f"best_path_length_file File: {summary['best_path_length_file']}, Score: {summary['best_path_length']}")
load_a_data(summary["best_path_length_file"])
print(f"best_node_count_file File: {summary['best_node_count_file']}, Score: {summary['best_node_count']}")
load_a_data(summary["best_node_count_file"])
print(f"best_iterations_file File: {summary['best_iterations_file']}, Score: {summary['best_iterations']}")
load_a_data(summary["best_iterations_file"])    


best_all_file File: None, Score: 1000000


TypeError: expected str, bytes or os.PathLike object, not NoneType

In [15]:
for index, row in best_param.iterrows():
    print(f"File: {row['parameter']}, BRRT_Optimize_score: {row['BRRT_Optimize_score']}, Diff: {row['Diff']}")
    show(data_agg[row['parameter']])
    display_data(data_summary_df[row['parameter']])


NameError: name 'best_param' is not defined